# Three-Way FRET Image Simulation — ALEX (488 / 561 nm)

Simulates Bayer-camera images of three-dye FRET triads (D → A1 → A2) under
alternating 488 nm and 561 nm excitation (ALEX), fits each frame with the
standard IRLS pipeline (`FittingStrategy.STANDARD_ITER`), and saves the
per-frame fit results (A_B, A_G, A_R, chi_sqr, …) together with the ground-truth
distances to HDF5 for subsequent distance-recovery analysis.

**Physics:**  
- 488 nm frame: D absorbs; FRET D→A1 (E1), D→A2 (E2), cascade A1→A2 (E12)  
- 561 nm frame: A1 absorbs; cascade A1→A2 (E12) only → directly resolves r12  

**See** `claude/FRET_3way_ImageSimulation.md` for the full plan.

In [ ]:
import sys, os, json, types, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import fpbase
from itertools import combinations
from pathlib import Path
from tqdm.notebook import tqdm

sys.path.insert(0, '../..')
import src.SpectralFunctions as SpectralFunctions
import src.Multicolour_Simulation_Functions as Multicolour_Simulation_Functions
import src.sCMOSFunctions as sCMOSFunctions
import src.MaskFunctions as MaskFunctions
import src.IOFunctions as IOFunctions
import src.CameraDefaults as CameraDefaults

# FittingStrategy must come from Multicolour_Simulation_Functions — _perform_fitting
# compares against its own local enum, not ImageAnalysisFunctions.FittingStrategy
from src.Multicolour_Simulation_Functions import FittingStrategy

SF    = SpectralFunctions.Spectral_Funcs()
MSF   = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()
sCMOS = sCMOSFunctions.sCMOS_Functions()
MF    = MaskFunctions.Mask_Functions()
IO    = IOFunctions.IO_Functions()

KAPPA_SQ = 2/3
N_REFR   = 1.33
print('Imports OK')
print(f'FittingStrategy.STANDARD_ITER = {FittingStrategy.STANDARD_ITER}')


## Camera calibration & spectral setup

In [ ]:
calib_dir = '../../Camera_Calibrations/ZWO_Camera'
gain_map     = IO.read_tiff(os.path.join(calib_dir, 'gain.tif'))
offset_map   = IO.read_tiff(os.path.join(calib_dir, 'offset.tif'))
variance_map = IO.read_tiff(os.path.join(calib_dir, 'variance.tif'))
read_noise   = IO.read_tiff(os.path.join(calib_dir, 'readnoise.tif'))
rqe_map      = IO.read_tiff(os.path.join(calib_dir, 'rqe.tif'))

GAIN_MED      = float(np.median(gain_map))
OFFSET_MED    = float(np.median(offset_map))
READNOISE_MED = float(np.median(read_noise))
VARIANCE_MED  = float(np.median(variance_map))

# ZWO camera defaults (pixel_size=78 nm, RGGB mosaic)
cam_cfg     = CameraDefaults.get_camera_config('zwo')
MOSAIC_UNIT = cam_cfg.mosaic_unit
PIXEL_SIZE  = cam_cfg.pixel_size * 1000   # µm → nm
NA          = 1.49

# Spectral QE curves from SF (shape of curves), normalised so peak = 0.91 (ZWO sensor peak QE)
R, G, B, wavelength = SF.getpixelefficiency()
pixel_QYs = (np.vstack([B, G, R]) / np.max(np.vstack([B, G, R]))) * 0.91

# Filter set: 515 nm longpass (matching 3way_FRET_simulation_with488.ipynb)
FILTERS = [
    'semrock-di03-r488-t1-25x36',
    'semrock-blp01-488r',
    'nikon-ti2-non-multiphoton-pfs-dichroic',
]

print(f'Camera: ZWO  gain={GAIN_MED:.3f} ADU/e⁻  offset={OFFSET_MED:.1f} ADU  '
      f'readnoise={READNOISE_MED:.2f} e⁻')
print(f'Pixel: {PIXEL_SIZE:.0f} nm  NA={NA}  mosaic={MOSAIC_UNIT}')
print(f'pixel_QYs: shape={pixel_QYs.shape}  peak={pixel_QYs.max():.3f}  '
      f'(normalised to 0.91)')


## Helper functions (Förster radii, dye RGB, triad metrics)

Copied verbatim from `3way_FRET_simulation_with488.ipynb`.

In [ ]:
def _get_absorption_data(dye_name):
    obj = fpbase.get_fluorophore(dye_name).default_state
    for subtype in ('AB', 'EX'):
        matches = [x for x in obj.spectra if subtype in x.subtype]
        if matches:
            ab_data = np.array(matches[0].data)
            ab_data[:, 1] = np.clip(ab_data[:, 1], 0, None)
            if subtype == 'EX':
                mx = ab_data[:, 1].max()
                if mx > 0:
                    ab_data[:, 1] /= mx
            return ab_data[:, 0], ab_data[:, 1]
    raise ValueError(f"No AB or EX spectrum for '{dye_name}'")


def absorption_peak_nm(dye_name):
    wl, vals = _get_absorption_data(dye_name)
    return float(wl[np.argmax(vals)])


def emission_peak_nm(dye_name):
    obj = fpbase.get_fluorophore(dye_name).default_state
    em  = [x for x in obj.spectra if 'EM' in x.subtype]
    if not em:
        return absorption_peak_nm(dye_name) + 20.0
    em_data = np.array(em[0].data)
    return float(em_data[np.argmax(em_data[:, 1]), 0])


def direct_excitation_frac(dye_name, laser_nm):
    wl, vals = _get_absorption_data(dye_name)
    mx = vals.max()
    if mx == 0:
        return 0.0
    return float(np.interp(laser_nm, wl, vals) / mx)


def get_FRET_pair(donor_dye, acceptor_dye):
    donor_obj  = fpbase.get_fluorophore(donor_dye).default_state
    em_data    = np.array([x for x in donor_obj.spectra if 'EM' in x.subtype][0].data)
    donor      = pd.DataFrame({'wavelength': em_data[:, 0],
                                'Emission (AU)': np.clip(em_data[:, 1], 0, None)})
    QY_D       = donor_obj.qy or 0.01
    acceptor_obj = fpbase.get_fluorophore(acceptor_dye).default_state
    QY_A         = acceptor_obj.qy or 0.01
    ab_matches = [x for x in acceptor_obj.spectra if 'AB' in x.subtype]
    ex_matches = [x for x in acceptor_obj.spectra if 'EX' in x.subtype]
    if ab_matches:
        ab_data = np.array(ab_matches[0].data)
        ab_data[:, 1] = np.clip(ab_data[:, 1], 0, None) * acceptor_obj.ext_coeff
    elif ex_matches:
        ab_data = np.array(ex_matches[0].data)
        ab_data[:, 1] = np.clip(ab_data[:, 1], 0, None)
        mx = ab_data[:, 1].max()
        if mx > 0:
            ab_data[:, 1] = ab_data[:, 1] / mx * acceptor_obj.ext_coeff
    else:
        raise ValueError(f"No AB/EX spectrum for '{acceptor_dye}'")
    acceptor = pd.DataFrame({'wavelength': ab_data[:, 0], 'absorption': ab_data[:, 1]})
    return acceptor, donor, QY_D, QY_A


def forster_radius_nm(donor_name, acceptor_name, n=N_REFR, kappa_sq=KAPPA_SQ):
    acceptor, donor, QY_D, QY_A = get_FRET_pair(donor_name, acceptor_name)
    wl_start = max(acceptor['wavelength'].iloc[0],  donor['wavelength'].iloc[0])
    wl_end   = min(acceptor['wavelength'].iloc[-1], donor['wavelength'].iloc[-1])
    donor    = donor[(donor['wavelength']    > wl_start) & (donor['wavelength']    < wl_end)].copy()
    acceptor = acceptor[(acceptor['wavelength'] > wl_start) & (acceptor['wavelength'] < wl_end)].copy()
    donor['Emission (norm)'] = donor['Emission (AU)'] / np.trapz(donor['Emission (AU)'])
    wl_m    = acceptor['wavelength'].to_numpy() * 1e-9
    overlap = acceptor['absorption'].to_numpy() * donor['Emission (norm)'].to_numpy()
    J       = np.trapz(overlap * wl_m ** 4)
    R0_m    = (2.11e-5) * (kappa_sq * n**(-4) * QY_D * J)**(1/6)
    return R0_m * 1e9, QY_D, QY_A


def dye_rgb(dye_name):
    """Return normalised [B, G, R] colour vector through the filter set."""
    _, rgb = SF.get_pixel_fractions_dye_and_filters(
        [dye_name], FILTERS, wavelength, pixel_QYs, normalized=True
    )
    rgb = np.asarray(rgb).ravel()
    return rgb / rgb.sum()


print('Helper functions defined.')

## Dye list → ALEX-practical triad selection

Criteria:
1. `practical`: both acceptors yield ≥ 500 photons at their own R₀  
2. `laser_practical` at **488 nm**: acceptors absorb < 15 % of their peak at 488 nm  
3. `alex_practical`: A1 absorbs ≥ 40 % of its peak at 561 nm (makes A1 the primary 561 nm absorber)  
4. `alex_clean`: D absorbs < 10 % of its peak at 561 nm, and A2 absorbs < 15 % at 561 nm  

In [ ]:
DYE_LIST = [
    'Alexa Fluor 488',
    'Cy3',
    'ATTO 550',
    'ATTO 647N',
    'Cy5',
    'ATTO 655',
    'Alexa Fluor 594',
]

LASER_WLS     = [488, 561]
I0            = 10_000   # unquenched donor photons (for metric calculation only)
N_MIN_PHOTONS = 500      # minimum acceptor photons at R0
ACC_DIRECT_LIMIT = 0.15  # max acceptor direct excitation fraction
DONOR_EXC_MIN    = 0.40  # min donor excitation fraction (viable donor line)
A1_561_MIN       = 0.40  # A1 must absorb ≥ 40% of its peak at 561 nm
D_561_MAX        = 0.10  # donor must absorb < 10% of its peak at 561 nm
A2_561_MAX       = 0.15  # A2 must absorb < 15% of its peak at 561 nm

print('Fetching absorption peaks and excitation fractions…')
peak_wl = {d: absorption_peak_nm(d) for d in DYE_LIST}
em_peak = {d: emission_peak_nm(d)   for d in DYE_LIST}
sorted_dyes = sorted(DYE_LIST, key=lambda d: peak_wl[d])
exc_table   = {d: {l: direct_excitation_frac(d, l) for l in LASER_WLS} for d in sorted_dyes}

print(f'\n{"Dye":20s}  {"peak_abs":>8s}  {"peak_em":>8s}  {"488nm":>7s}  {"561nm":>7s}')
print('-'*60)
for d in sorted_dyes:
    print(f'{d:20s}  {peak_wl[d]:8.0f}  {em_peak[d]:8.0f}  '
          f'{exc_table[d][488]*100:6.1f}%  {exc_table[d][561]*100:6.1f}%')

# Enumerate all triads with ≥ 2 red-shifted acceptors
all_triads = []
for donor in sorted_dyes:
    acceptors = [d for d in sorted_dyes if peak_wl[d] > peak_wl[donor]]
    if len(acceptors) < 2:
        continue
    for a1, a2 in combinations(acceptors, 2):
        all_triads.append((donor, a1, a2))

print(f'\nTotal triads: {len(all_triads)}')

In [ ]:
def triad_photons(r1, r2, R0_DA1, R0_DA2, QY_D, QY_A1, QY_A2, I0=I0):
    k1    = (r1/R0_DA1)**(-6)
    k2    = (r2/R0_DA2)**(-6)
    denom = 1 + k1 + k2
    E1    = k1 / denom
    E2    = k2 / denom
    N_D   = I0 * (1 - E1 - E2)
    N_A1  = I0 * (E1 / QY_D) * QY_A1
    N_A2  = I0 * (E2 / QY_D) * QY_A2
    return N_D, N_A1, N_A2


print('Computing Förster radii and triad metrics…')
results = []
for donor, a1, a2 in all_triads:
    try:
        R0_DA1, QY_D,  QY_A1 = forster_radius_nm(donor, a1)
        R0_DA2, _,     QY_A2 = forster_radius_nm(donor, a2)
        R0_A1A2, _,    _     = forster_radius_nm(a1,    a2)
    except Exception as e:
        print(f'  Skip {donor}→{a1}+{a2}: {e}')
        continue

    rgb_D  = dye_rgb(donor)
    rgb_A1 = dye_rgb(a1)
    rgb_A2 = dye_rgb(a2)

    # Photon counts at respective R0 (metric: are acceptors bright enough?)
    _, N_A1_at_R0, _ = triad_photons(R0_DA1, 10*R0_DA2, R0_DA1, R0_DA2, QY_D, QY_A1, QY_A2)
    _, _, N_A2_at_R0 = triad_photons(10*R0_DA1, R0_DA2, R0_DA1, R0_DA2, QY_D, QY_A1, QY_A2)
    practical = bool(min(N_A1_at_R0, N_A2_at_R0) >= N_MIN_PHOTONS)

    # 488 nm laser: check acceptor direct excitation
    viable_488 = exc_table[donor][488] >= DONOR_EXC_MIN
    laser_practical = bool(
        viable_488 and
        exc_table[a1][488] <= ACC_DIRECT_LIMIT and
        exc_table[a2][488] <= ACC_DIRECT_LIMIT
    )

    # 561 nm ALEX criterion
    alex_practical = bool(
        exc_table[a1][561] >= A1_561_MIN and
        exc_table[donor][561] <= D_561_MAX and
        exc_table[a2][561] <= A2_561_MAX
    )

    results.append(dict(
        donor=donor, acc1=a1, acc2=a2,
        R0_DA1=round(R0_DA1,2), R0_DA2=round(R0_DA2,2), R0_A1A2=round(R0_A1A2,2),
        QY_D=QY_D, QY_A1=QY_A1, QY_A2=QY_A2,
        N_A1_at_R0=round(N_A1_at_R0), N_A2_at_R0=round(N_A2_at_R0),
        exc_a1_488=round(exc_table[a1][488],3), exc_a2_488=round(exc_table[a2][488],3),
        exc_a1_561=round(exc_table[a1][561],3), exc_a2_561=round(exc_table[a2][561],3),
        exc_d_561=round(exc_table[donor][561],3),
        practical=practical,
        laser_practical=laser_practical,
        alex_practical=alex_practical,
        fully_practical=bool(practical and laser_practical and alex_practical),
        _R0_DA1=R0_DA1, _R0_DA2=R0_DA2, _R0_A1A2=R0_A1A2,
        _QY_D=QY_D, _QY_A1=QY_A1, _QY_A2=QY_A2,
        _rgb_D=rgb_D, _rgb_A1=rgb_A1, _rgb_A2=rgb_A2,
        _em_D=em_peak[donor], _em_A1=em_peak[a1], _em_A2=em_peak[a2],
    ))

results_df = pd.DataFrame(results)
alex_rows  = [r for r in results if r['fully_practical']]
print(f'\nTotal triads evaluated: {len(results)}')
print(f'ALEX-practical triads:  {len(alex_rows)}')

In [ ]:
display_cols = ['donor','acc1','acc2',
                'R0_DA1','R0_DA2','R0_A1A2',
                'QY_D','QY_A1','QY_A2',
                'exc_a1_488','exc_a2_488','exc_a1_561','exc_a2_561','exc_d_561',
                'practical','laser_practical','alex_practical','fully_practical']

def _flag_false(v):
    return 'color:red;font-weight:bold' if v is False else ''
def _flag_true(v):
    return 'color:green;font-weight:bold' if v is True else ''

(results_df[display_cols]
 .sort_values('fully_practical', ascending=False)
 .style
 .map(_flag_false, subset=['practical','laser_practical','alex_practical','fully_practical'])
 .map(_flag_true,  subset=['fully_practical'])
 .format({'R0_DA1':'{:.1f} nm','R0_DA2':'{:.1f} nm','R0_A1A2':'{:.1f} nm',
          'QY_D':'{:.2f}','QY_A1':'{:.2f}','QY_A2':'{:.2f}',
          'exc_a1_488':'{:.1%}','exc_a2_488':'{:.1%}',
          'exc_a1_561':'{:.1%}','exc_a2_561':'{:.1%}','exc_d_561':'{:.1%}'}))

## Cascade FRET photon budget functions

In [ ]:
def fret_efficiencies(r1, r2, r12, R0_DA1, R0_DA2, R0_A1A2):
    """Return (E1, E2, E12) for given distances."""
    k1  = (r1  / R0_DA1 )**(-6)
    k2  = (r2  / R0_DA2 )**(-6)
    k12 = (r12 / R0_A1A2)**(-6)
    E1  = k1  / (1 + k1 + k2)
    E2  = k2  / (1 + k1 + k2)
    E12 = k12 / (1 + k12)
    return E1, E2, E12


def photons_488(r1, r2, r12, R0_DA1, R0_DA2, R0_A1A2,
                QY_D, QY_A1, QY_A2, I0_D,
                delta_A1_488=0.0, delta_A2_488=0.0):
    """Mean photon counts for 488 nm excitation with A1→A2 cascade.

    Args:
        delta_A1_488: ε_A1(488)/ε_D(488) — relative direct excitation of A1
        delta_A2_488: ε_A2(488)/ε_D(488) — relative direct excitation of A2

    Returns:
        (mu_D, mu_A1, mu_A2)  expected photon counts
    """
    E1, E2, E12 = fret_efficiencies(r1, r2, r12, R0_DA1, R0_DA2, R0_A1A2)
    excitations = I0_D / QY_D   # donor excitations per frame
    mu_D   = I0_D * (1 - E1 - E2)
    mu_A1  = (excitations * E1 * QY_A1 * (1 - E12)
              + excitations * delta_A1_488 * QY_A1)
    mu_A2  = (excitations * (E2 + E1 * E12) * QY_A2
              + excitations * delta_A2_488 * QY_A2)
    return mu_D, mu_A1, mu_A2


def photons_561(r12, R0_A1A2, QY_A1, QY_A2, I0_A1,
                delta_A2_561=0.0):
    """Mean photon counts for 561 nm excitation (A1 is primary absorber).

    Args:
        I0_A1: unquenched A1 photon budget under 561 nm excitation
        delta_A2_561: ε_A2(561)/ε_A1(561) — relative direct excitation of A2

    Returns:
        (mu_A1_561, mu_A2_561)  expected photon counts (mu_D_561 ≈ 0)
    """
    _, _, E12 = fret_efficiencies(1.0, 1.0, r12, 1.0, 1.0, R0_A1A2)   # only E12 used
    excitations_A1 = I0_A1 / QY_A1
    mu_A1 = I0_A1 * (1 - E12)
    mu_A2 = excitations_A1 * E12 * QY_A2 + excitations_A1 * delta_A2_561 * QY_A2
    return mu_A1, mu_A2


# Sanity check: no cascade limit matches existing notebook
_m = results[0] if results else None
if _m:
    mu_D, mu_A1, mu_A2 = photons_488(
        _m['_R0_DA1'], _m['_R0_DA2'], 1000.0,
        _m['_R0_DA1'], _m['_R0_DA2'], 1000.0,   # r12=1000 nm → E12≈0
        _m['_QY_D'], _m['_QY_A1'], _m['_QY_A2'], I0_D=I0
    )
    # At r1=R0, r2=R0, r12→∞: E1=E2=1/3, E12=0
    print(f'Sanity (r1=r2=R0, no cascade): N_D={mu_D:.0f}  N_A1={mu_A1:.0f}  N_A2={mu_A2:.0f}')
    print(f'Expected E1=E2≈0.333 → N_D≈{I0/3:.0f}')

## Simulation & fitting setup

In [ ]:
from src.Multicolour_Simulation_Functions import SimulationConfig, CameraParameters

# ── Simulation parameters — stored in SimulationConfig so MSF methods receive them ──
CROP_SZ  = 14      # pixels (square crop)
I0_D     = 2000    # unquenched donor photons per 488 nm frame
I0_A1    = 2000    # unquenched A1 photons per 561 nm frame
BG_PE_PX = 3.0     # background per pixel (pe)
N_SIM    = 5000    # Monte Carlo trials per (r1, r2, r12, laser)

APPLY_TRIANGLE_FILTER = True   # enforce |r1-r2| ≤ r12 ≤ r1+r2

# Distance grid: 20 points from 0.2 R0 to 2.0 R0 for each axis
RHO1_VALS  = np.linspace(0.2, 2.0, 20)   # r1  / R0_DA1
RHO2_VALS  = np.linspace(0.2, 2.0, 20)   # r2  / R0_DA2
RHO12_VALS = np.linspace(0.2, 2.0, 20)   # r12 / R0_A1A2

# SimulationConfig carries NA, pixel_size, n_bootstrap and background to MSF internals
sim_config = SimulationConfig(
    n_bootstrap=N_SIM,
    NA=1.49,
    pixel_size=PIXEL_SIZE,              # nm
    background_photons=3.0 * BG_PE_PX, # gen_camera_image_stack splits bg across 3 channels
    save_raw_results=False,
    use_stochastic_photons=False,       # we pre-draw Poisson outside MSF
)

# Camera params dict — validated once into a CameraParameters dataclass for MSF methods
masks_crop = MF.get_masks(size_x=CROP_SZ, size_y=CROP_SZ, mosaic_unit=MOSAIC_UNIT)

camera_params_dict = {
    'gain':                np.full((CROP_SZ, CROP_SZ), GAIN_MED),
    'offset':              np.full((CROP_SZ, CROP_SZ), OFFSET_MED),
    'variance':            np.full((CROP_SZ, CROP_SZ), VARIANCE_MED),
    'readnoise':           np.full((CROP_SZ, CROP_SZ), READNOISE_MED),
    'rqe':                 np.full((CROP_SZ, CROP_SZ), 1.0),
    'pixel_QYs':           pixel_QYs,
    'pixel_order':         ['B', 'G', 'R'],
    'pixel_order_indices': [0, 1, 2],
    'masks':               masks_crop,
}
cam_params = CameraParameters.validate_and_create(camera_params_dict)

smoothing_fn = types.SimpleNamespace(
    args={'sigma': 1.5},
    extent=1.5,
    smoothing_function=sCMOS.gaussian_filter_stack,
    data_arg='image',
)

# Molecule fixed at crop centre; x0y0 shape (N_SIM, 2, 1)
MOL_X_NM = CROP_SZ / 2.0 * PIXEL_SIZE
MOL_Y_NM = CROP_SZ / 2.0 * PIXEL_SIZE

def _make_x0y0(n_frames):
    x0y0 = {'mol': np.zeros((n_frames, 2, 1))}
    x0y0['mol'][:, 0, 0] = MOL_X_NM
    x0y0['mol'][:, 1, 0] = MOL_Y_NM
    return x0y0

n_configs_before_filter = len(RHO1_VALS) * len(RHO2_VALS) * len(RHO12_VALS)
print(f'Crop: {CROP_SZ}×{CROP_SZ} px  molecule at ({MOL_X_NM:.0f}, {MOL_Y_NM:.0f}) nm')
print(f'I0_D={I0_D}  I0_A1={I0_A1}  N_SIM={N_SIM}  triangle_filter={APPLY_TRIANGLE_FILTER}')
print(f'Grid: 20×20×20 = {n_configs_before_filter:,} configs before triangle filter')
print(f'  ρ range: {RHO1_VALS[0]:.2f} – {RHO1_VALS[-1]:.2f} R0  (same for all three axes)')


## Core simulation + fitting functions

In [ ]:
def simulate_and_fit(pe_frames, lambda_avg_frames, n_photons_arr):
    """Simulate N Bayer frames from pre-drawn Poisson photon counts, then fit.

    Uses MSF._prepare_fitting_data / _compute_error_maps / _perform_fitting so the
    pipeline is identical to test_simulation_method — including the sqrt error
    correction applied in _fit_standard_iter.

    Args:
        pe_frames:         (N, 3) normalised [B,G,R] composite pe vectors
        lambda_avg_frames: (N,) emission-weighted mean wavelength in nm
        n_photons_arr:     (N,) total photon counts (Poisson-drawn before call)

    Returns:
        pd.DataFrame with xc,yc,s_x,s_y,bg_B,bg_G,bg_R,A_B,A_G,A_R,chi_sqr,
                          *_err columns, photons, A_B_norm, A_G_norm, A_R_norm
    """
    n_frames = len(n_photons_arr)
    x0y0 = _make_x0y0(n_frames)

    # Step 1: simulate — gen_camera_image_stack returns (bayer, smoothed, normal)
    bayer_stack, smoothed_stack, _ = MSF.gen_camera_image_stack(
        camera_params_dict,
        wavelength,
        lambda_avg_frames,        # nm, matches pixel_size units
        pe_frames,                # (N,3) stochastic spectral mode
        {'mol': n_photons_arr.astype(float)},
        x0y0,
        smoothing_function=smoothing_fn,
        background_photons=sim_config.background_photons,
        NA=sim_config.NA,
        pixel_size=sim_config.pixel_size,
    )

    # gen_camera_image_stack squeezes the output when N=1: restore the batch dimension
    # so _prepare_fitting_data and _fit_standard_iter always see (N, H, W)
    if bayer_stack.ndim == 2:
        bayer_stack   = bayer_stack[np.newaxis]
        smoothed_stack = smoothed_stack[np.newaxis]

    # Step 2: convert ADU→pe (reuses smoothed_stack from gen_camera_image_stack)
    # Use a per-call config with the correct n_bootstrap so _fit_standard_iter
    # iterates over exactly the frames we simulated, not the global N_SIM default
    call_config = SimulationConfig(
        n_bootstrap=n_frames,
        NA=sim_config.NA,
        pixel_size=sim_config.pixel_size,
        background_photons=sim_config.background_photons,
        save_raw_results=False,
        use_stochastic_photons=False,
    )

    pe_data, sm_data, _ = MSF._prepare_fitting_data(
        bayer_stack, smoothed_stack, cam_params, FittingStrategy.STANDARD_ITER, call_config
    )

    # Step 3: compute inverse-variance weights from smoothed pe data
    weights_map, _ = MSF._compute_error_maps(sm_data, None, cam_params)

    # Step 4: fit
    df = MSF._perform_fitting(
        FittingStrategy.STANDARD_ITER, pe_data, sm_data, weights_map, None,
        cam_params, call_config
    )
    return df


print('simulate_and_fit defined.')


## Single-configuration demo

Quick sanity check on the best triad at r1 = r2 = r12 = R0.

In [ ]:
if not alex_rows:
    print('No ALEX-practical triads found — widen criteria or add dyes.')
else:
    m = alex_rows[0]
    rng = np.random.default_rng(42)

    R0_DA1, R0_DA2, R0_A1A2 = m['_R0_DA1'], m['_R0_DA2'], m['_R0_A1A2']
    QY_D,   QY_A1,  QY_A2   = m['_QY_D'],   m['_QY_A1'],  m['_QY_A2']
    rgb_D,  rgb_A1, rgb_A2  = m['_rgb_D'],  m['_rgb_A1'], m['_rgb_A2']
    em_D,   em_A1,  em_A2   = m['_em_D'],   m['_em_A1'],  m['_em_A2']

    d_A1_488 = exc_table[m['acc1']][488] / max(exc_table[m['donor']][488], 1e-6)
    d_A2_488 = exc_table[m['acc2']][488] / max(exc_table[m['donor']][488], 1e-6)
    d_A2_561 = exc_table[m['acc2']][561] / max(exc_table[m['acc1']][561], 1e-6)

    r1_demo, r2_demo, r12_demo = R0_DA1, R0_DA2, R0_A1A2

    def _draw_composite(mu_D, mu_A1, mu_A2, rgb_D, rgb_A1, rgb_A2, em_D, em_A1, em_A2, n=1):
        """Draw Poisson photon counts and return (pe_frames, lam_frames, n_tot)."""
        N_D  = rng.poisson(max(mu_D,  0), size=n).astype(float)
        N_A1 = rng.poisson(max(mu_A1, 0), size=n).astype(float)
        N_A2 = rng.poisson(max(mu_A2, 0), size=n).astype(float)
        rgb  = N_D[:,None]*rgb_D + N_A1[:,None]*rgb_A1 + N_A2[:,None]*rgb_A2
        N_tot = rgb.sum(axis=1)
        pe    = rgb / np.maximum(N_tot[:,None], 1e-9)
        lam   = (N_D*em_D + N_A1*em_A1 + N_A2*em_A2) / np.maximum(N_tot, 1e-9)
        return pe, lam, N_tot

    # ── 488 nm: one demo frame ────────────────────────────────────────────────
    mu_D, mu_A1, mu_A2 = photons_488(
        r1_demo, r2_demo, r12_demo, R0_DA1, R0_DA2, R0_A1A2,
        QY_D, QY_A1, QY_A2, I0_D, delta_A1_488=d_A1_488, delta_A2_488=d_A2_488
    )
    pe_488, lam_488, n_488 = _draw_composite(mu_D, mu_A1, mu_A2, rgb_D, rgb_A1, rgb_A2, em_D, em_A1, em_A2)
    df_demo_488 = simulate_and_fit(pe_488, lam_488, n_488)

    # ── 561 nm: one demo frame ────────────────────────────────────────────────
    mu_A1_561, mu_A2_561 = photons_561(r12_demo, R0_A1A2, QY_A1, QY_A2, I0_A1, delta_A2_561=d_A2_561)
    pe_561, lam_561, n_561 = _draw_composite(0, mu_A1_561, mu_A2_561, rgb_D, rgb_A1, rgb_A2, em_D, em_A1, em_A2)
    df_demo_561 = simulate_and_fit(pe_561, lam_561, n_561)

    # ── Reconstruct one Bayer image for display (simulate_and_fit doesn't expose bayer) ──
    def _one_bayer(pe, lam, n_tot):
        bayer, _, _ = MSF.gen_camera_image_stack(
            camera_params_dict, wavelength, lam, pe,
            {'mol': n_tot.astype(float)}, _make_x0y0(len(n_tot)),
            smoothing_function=smoothing_fn,
            background_photons=sim_config.background_photons,
            NA=sim_config.NA, pixel_size=sim_config.pixel_size,
        )
        return bayer[0]

    bayer_488_img = _one_bayer(pe_488, lam_488, n_488)
    bayer_561_img = _one_bayer(pe_561, lam_561, n_561)

    fig, axs = plt.subplots(1, 2, figsize=(7, 3))
    for ax, bayer, title in [
        (axs[0], bayer_488_img, f'488 nm  {m["donor"]}→{m["acc1"]}+{m["acc2"]}'),
        (axs[1], bayer_561_img, f'561 nm  {m["acc1"]}→{m["acc2"]} cascade'),
    ]:
        ax.imshow(bayer, cmap='gray', origin='lower',
                  vmin=np.percentile(bayer, 1), vmax=np.percentile(bayer, 99.5))
        ax.set_title(title, fontsize=9); ax.set_xlabel('x (px)'); ax.set_ylabel('y (px)')
    plt.tight_layout(); plt.show()

    print(f'Best ALEX triad: {m["donor"]} → {m["acc1"]} + {m["acc2"]}')
    print(f'  R0(D-A1)={R0_DA1:.1f} nm  R0(D-A2)={R0_DA2:.1f} nm  R0(A1-A2)={R0_A1A2:.1f} nm')
    print(f'  488 nm:  μ_D={mu_D:.0f}  μ_A1={mu_A1:.0f}  μ_A2={mu_A2:.0f}')
    print(f'  561 nm:  μ_A1={mu_A1_561:.0f}  μ_A2={mu_A2_561:.0f}')
    print(f'  Demo fit (488): A_B={df_demo_488["A_B"].iloc[0]:.1f}  '
          f'A_G={df_demo_488["A_G"].iloc[0]:.1f}  A_R={df_demo_488["A_R"].iloc[0]:.1f}')


## Main simulation loop — all ALEX-practical triads

For each triad and each (r1, r2, r12) configuration:
1. Simulate `N_SIM` Bayer frames under 488 nm excitation  
2. Simulate `N_SIM` Bayer frames under 561 nm excitation  
3. Fit each stack with `STANDARD_ITER`  
4. Record ground-truth distances, FRET efficiencies, expected photons, and fit results  

In [ ]:
def _draw_composite_frames(mu_D, mu_A1, mu_A2, rgb_D, rgb_A1, rgb_A2,
                            em_D, em_A1, em_A2, n, rng):
    """Poisson-draw N photon counts per dye and build composite (pe, lam, N_tot) arrays."""
    N_D  = rng.poisson(max(mu_D,  0), size=n).astype(float)
    N_A1 = rng.poisson(max(mu_A1, 0), size=n).astype(float)
    N_A2 = rng.poisson(max(mu_A2, 0), size=n).astype(float)
    rgb  = N_D[:,None]*rgb_D + N_A1[:,None]*rgb_A1 + N_A2[:,None]*rgb_A2  # (n,3)
    N_tot = rgb.sum(axis=1)
    good  = N_tot > 5
    pe    = np.where(good[:,None], rgb / np.maximum(N_tot[:,None], 1e-9), np.ones((n,3))/3)
    lam   = np.where(N_tot > 0,
                     (N_D*em_D + N_A1*em_A1 + N_A2*em_A2) / np.maximum(N_tot, 1e-9),
                     em_D * np.ones(n))
    return pe, lam, N_tot, N_D, N_A1, N_A2


def run_simulation_for_triad(m, rng):
    """Simulate + fit all distance configurations for one triad.

    Returns a flat DataFrame with ground truth + fit results for all configs
    and both laser excitations.
    """
    R0_DA1, R0_DA2, R0_A1A2 = m['_R0_DA1'], m['_R0_DA2'], m['_R0_A1A2']
    QY_D,   QY_A1,  QY_A2   = m['_QY_D'],   m['_QY_A1'],  m['_QY_A2']
    rgb_D,  rgb_A1, rgb_A2  = m['_rgb_D'],  m['_rgb_A1'], m['_rgb_A2']
    em_D,   em_A1,  em_A2   = m['_em_D'],   m['_em_A1'],  m['_em_A2']

    d_A1_488 = exc_table[m['acc1']][488] / max(exc_table[m['donor']][488], 1e-9)
    d_A2_488 = exc_table[m['acc2']][488] / max(exc_table[m['donor']][488], 1e-9)
    d_A2_561 = exc_table[m['acc2']][561] / max(exc_table[m['acc1']][561], 1e-9)

    configs = [
        (r1, r2, r12)
        for r1  in RHO1_VALS  * R0_DA1
        for r2  in RHO2_VALS  * R0_DA2
        for r12 in RHO12_VALS * R0_A1A2
        if not APPLY_TRIANGLE_FILTER or (abs(r1 - r2) <= r12 <= r1 + r2)
    ]

    all_dfs = []

    for r1, r2, r12 in tqdm(configs, desc=f'{m["donor"]}→{m["acc1"]}+{m["acc2"]}', leave=False):
        E1, E2, E12 = fret_efficiencies(r1, r2, r12, R0_DA1, R0_DA2, R0_A1A2)

        # ── 488 nm ────────────────────────────────────────────────────────────
        mu_D, mu_A1, mu_A2 = photons_488(
            r1, r2, r12, R0_DA1, R0_DA2, R0_A1A2,
            QY_D, QY_A1, QY_A2, I0_D, delta_A1_488=d_A1_488, delta_A2_488=d_A2_488
        )
        pe, lam, n_tot, N_D_draw, N_A1_draw, N_A2_draw = _draw_composite_frames(
            mu_D, mu_A1, mu_A2, rgb_D, rgb_A1, rgb_A2, em_D, em_A1, em_A2, N_SIM, rng
        )
        df_488 = simulate_and_fit(pe, lam, n_tot)
        df_488['laser_nm'] = 488
        df_488['mu_D']     = mu_D;  df_488['mu_A1']    = mu_A1;  df_488['mu_A2']    = mu_A2
        df_488['N_D_draw'] = N_D_draw; df_488['N_A1_draw'] = N_A1_draw; df_488['N_A2_draw'] = N_A2_draw

        # ── 561 nm ────────────────────────────────────────────────────────────
        mu_A1_561, mu_A2_561 = photons_561(
            r12, R0_A1A2, QY_A1, QY_A2, I0_A1, delta_A2_561=d_A2_561
        )
        pe, lam, n_tot, _, N_A1_561_draw, N_A2_561_draw = _draw_composite_frames(
            0, mu_A1_561, mu_A2_561, rgb_D, rgb_A1, rgb_A2, em_D, em_A1, em_A2, N_SIM, rng
        )
        df_561 = simulate_and_fit(pe, lam, n_tot)
        df_561['laser_nm'] = 561
        df_561['mu_D']     = 0.0;  df_561['mu_A1']    = mu_A1_561; df_561['mu_A2']    = mu_A2_561
        df_561['N_D_draw'] = 0.0;  df_561['N_A1_draw'] = N_A1_561_draw; df_561['N_A2_draw'] = N_A2_561_draw

        # ── Ground truth ──────────────────────────────────────────────────────
        for df in (df_488, df_561):
            df['r1_nm']  = r1;  df['r2_nm']  = r2;  df['r12_nm'] = r12
            df['E1']     = E1;  df['E2']     = E2;  df['E12']    = E12
            df['rho1']   = r1/R0_DA1; df['rho2'] = r2/R0_DA2; df['rho12'] = r12/R0_A1A2

        all_dfs.append(pd.concat([df_488, df_561], ignore_index=True))

    return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()


print('Main loop function defined. Ready to run.')


In [ ]:
# Output directory
OUT_DIR = Path('/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/FRET_3way_ALEX')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Saving results to: {OUT_DIR}')

rng_main = np.random.default_rng(2026)

triad_summaries = []

for m in alex_rows:
    triad_label = f'{m["donor"]}_{m["acc1"]}_{m["acc2"]}'
    safe_label  = triad_label.replace(' ','_').replace('/','_')
    h5_path     = OUT_DIR / f'{safe_label}.h5'
    json_path   = OUT_DIR / f'{safe_label}_metadata.json'

    print(f'\n=== {triad_label} ===')

    df_triad = run_simulation_for_triad(m, rng_main)

    if df_triad.empty:
        print('  No valid configurations — skipping.')
        continue

    # Drop all-NaN fit rows (failed fits)
    fit_cols = ['A_B','A_G','A_R']
    df_clean = df_triad.dropna(subset=fit_cols, how='all').copy()
    n_total  = len(df_triad)
    n_fit    = len(df_clean)
    print(f'  Fit success: {n_fit}/{n_total} ({100*n_fit/max(n_total,1):.1f}%)')

    # Normalise photon columns and write HDF5
    IO.write_h5_database(df_clean, str(h5_path), append=False, normalise_photons=True)
    print(f'  Saved {len(df_clean):,} rows → {h5_path.name}')

    # Save metadata JSON alongside
    meta = {
        'donor': m['donor'], 'acc1': m['acc1'], 'acc2': m['acc2'],
        'R0_DA1_nm': m['_R0_DA1'], 'R0_DA2_nm': m['_R0_DA2'], 'R0_A1A2_nm': m['_R0_A1A2'],
        'QY_D': m['_QY_D'], 'QY_A1': m['_QY_A1'], 'QY_A2': m['_QY_A2'],
        'rgb_D': m['_rgb_D'].tolist(), 'rgb_A1': m['_rgb_A1'].tolist(),
        'rgb_A2': m['_rgb_A2'].tolist(),
        'I0_D': I0_D, 'I0_A1': I0_A1, 'N_SIM': N_SIM,
        'pixel_size_nm': PIXEL_SIZE, 'NA': NA,
        'rho1_vals': RHO1_VALS.tolist(), 'rho2_vals': RHO2_VALS.tolist(),
        'rho12_vals': RHO12_VALS.tolist(),
        'triangle_filter': APPLY_TRIANGLE_FILTER,
        'fitting_strategy': 'STANDARD_ITER',
        'n_configs': len(df_triad[['r1_nm','r2_nm','r12_nm']].drop_duplicates()),
        'fit_success_rate': float(n_fit / max(n_total, 1)),
    }
    with open(json_path, 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'  Metadata → {json_path.name}')

    triad_summaries.append({'triad': triad_label, 'n_fit': n_fit, 'n_total': n_total,
                            'success_rate': n_fit/max(n_total,1)})

print('\n=== All triads complete ===')
pd.DataFrame(triad_summaries)

## QC plots

For the best triad: visualise fit quality, photon distributions, and colour-ratio
separation between 488 nm and 561 nm frame pairs.

In [ ]:
if alex_rows:
    m0        = alex_rows[0]
    safe0     = f'{m0["donor"]}_{m0["acc1"]}_{m0["acc2"]}'.replace(' ','_').replace('/','_')
    h5_path0  = OUT_DIR / f'{safe0}.h5'

    if h5_path0.exists():
        df_qc = IO.read_h5(str(h5_path0))

        # Select one configuration near R0 for each axis
        rho_target = 1.0
        mask_config = (
            (np.abs(df_qc['rho1']  - rho_target) < 0.15) &
            (np.abs(df_qc['rho2']  - rho_target) < 0.15) &
            (np.abs(df_qc['rho12'] - rho_target) < 0.15)
        )
        df_ref = df_qc[mask_config].copy()
        df_488_ref = df_ref[df_ref['laser_nm'] == 488]
        df_561_ref = df_ref[df_ref['laser_nm'] == 561]

        fig, axs = plt.subplots(2, 3, figsize=(12, 7))
        fig.suptitle(f'{m0["donor"]} → {m0["acc1"]} + {m0["acc2"]}  '
                     f'(ρ1=ρ2=ρ12≈1)',
                     fontsize=11)

        # Row 0: 488 nm distributions
        for ax, col, color, label in [
            (axs[0,0], 'A_B', '#5599ff', 'A_B'),
            (axs[0,1], 'A_G', '#33cc33', 'A_G'),
            (axs[0,2], 'A_R', '#ee3333', 'A_R'),
        ]:
            vals = df_488_ref[col].dropna()
            ax.hist(vals, bins=30, color=color, alpha=0.75, density=True)
            ax.set_title(f'488 nm  {label}  (N={len(vals)})', fontsize=9)
            ax.set_xlabel('Amplitude (pe)')

        # Row 1: 561 nm distributions + A_R_norm scatter
        for ax, col, color, label in [
            (axs[1,0], 'A_B', '#5599ff', 'A_B'),
            (axs[1,1], 'A_G', '#33cc33', 'A_G'),
            (axs[1,2], 'A_R', '#ee3333', 'A_R'),
        ]:
            vals = df_561_ref[col].dropna()
            ax.hist(vals, bins=30, color=color, alpha=0.75, density=True)
            ax.set_title(f'561 nm  {label}  (N={len(vals)})', fontsize=9)
            ax.set_xlabel('Amplitude (pe)')

        plt.tight_layout()
        plt.savefig(OUT_DIR / f'{safe0}_QC_distributions.pdf', dpi=150, bbox_inches='tight')
        plt.show()

        # ── Colour-ratio scatter: A_R_norm vs A_G_norm coloured by rho12 ──────
        fig2, axs2 = plt.subplots(1, 2, figsize=(10, 4))
        for ax, df_sub, title in [
            (axs2[0], df_qc[df_qc['laser_nm']==488], '488 nm excitation'),
            (axs2[1], df_qc[df_qc['laser_nm']==561], '561 nm excitation'),
        ]:
            sc = ax.scatter(
                df_sub['A_G_norm'], df_sub['A_R_norm'],
                c=df_sub['rho12'], cmap='viridis', s=4, alpha=0.4,
                vmin=RHO12_VALS.min(), vmax=RHO12_VALS.max()
            )
            plt.colorbar(sc, ax=ax, label='ρ12 = r12/R0(A1-A2)')
            ax.set_xlabel('A_G_norm'); ax.set_ylabel('A_R_norm')
            ax.set_title(title, fontsize=9)

        fig2.suptitle(f'{m0["donor"]} → {m0["acc1"]} + {m0["acc2"]}', fontsize=10)
        plt.tight_layout()
        plt.savefig(OUT_DIR / f'{safe0}_QC_colourratios.pdf', dpi=150, bbox_inches='tight')
        plt.show()

        # ── Chi-squared distribution ───────────────────────────────────────────
        fig3, ax3 = plt.subplots(figsize=(5, 3))
        for df_sub, color, label in [
            (df_qc[df_qc['laser_nm']==488], 'steelblue', '488 nm'),
            (df_qc[df_qc['laser_nm']==561], 'tomato',    '561 nm'),
        ]:
            chi2 = df_sub['chi_sqr'].dropna()
            ax3.hist(chi2.clip(0, 5), bins=50, color=color, alpha=0.6, density=True, label=label)
        ax3.axvline(1.0, color='k', ls='--', lw=1, label='χ²=1')
        ax3.set_xlabel('χ²'); ax3.set_ylabel('density'); ax3.legend(fontsize=8)
        ax3.set_title('Fit quality (all configs)', fontsize=9)
        plt.tight_layout()
        plt.savefig(OUT_DIR / f'{safe0}_QC_chisqr.pdf', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print(f'HDF5 not found at {h5_path0}  (run the loop cell first)')

## Summary

One HDF5 file per ALEX-practical triad is saved to `OUT_DIR`.  
Each file contains a flat table with columns:

| Group | Columns |
|-------|---------|
| Ground truth | `r1_nm`, `r2_nm`, `r12_nm`, `E1`, `E2`, `E12`, `rho1`, `rho2`, `rho12` |
| Excitation label | `laser_nm` (488 or 561) |
| Expected photons | `mu_D`, `mu_A1`, `mu_A2` |
| Poisson draws | `N_D_draw`, `N_A1_draw`, `N_A2_draw` |
| Fit (raw) | `xc`, `yc`, `s_x`, `s_y`, `bg_B`, `bg_G`, `bg_R`, `A_B`, `A_G`, `A_R`, `chi_sqr` |
| Fit (derived) | `photons`, `A_B_norm`, `A_G_norm`, `A_R_norm` (added by `write_h5_database`) |
| Fit errors | `xc_err`, `yc_err`, …, `A_R_err` |

A companion `{triad}_metadata.json` stores the Förster radii, QYs, RGB vectors,
and simulation parameters for use by the analysis notebook.

**Next:** open `notebooks/fret/3way_FRET_DistanceRecovery.ipynb` (to be written)
to load these files and perform distance unmixing / degeneracy analysis.